In [ ]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")


import os
import re
import json
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

DOMAINS = ["A", "B", "C", "D", "E"]

# ---- NEU: Output-Contract für EINEN Request mit A..E ----
OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "A": {"severity": <int>},
  "B": {"severity": <int>},
  "C": {"severity": <int>},
  "D": {"severity": <int>},
  "E": {"severity": <int>}
}
""".strip()


def _safe_model_name(s: str) -> str:
    return re.sub(r"[^\w\.-]+", "-", s)

def _temp_tag(t: float) -> str:
    if float(t).is_integer():
        return f"t{int(t)}"
    return f"t{str(t).replace('.', '_')}"

def next_free_run_path(results_dir: Path, model: str, temperature: float) -> Path:
    results_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in results_dir.glob("run_*.csv"):
        m = re.search(r"run_(\d+)_", p.name)
        if m:
            nums.append(int(m.group(1)))
    n = (max(nums) + 1) if nums else 1
    return results_dir / f"run_{n}_{_safe_model_name(model)}_{_temp_tag(temperature)}.csv"


def _extract_text_from_anthropic_message(message_obj: dict) -> str:
    parts = []
    for c in message_obj.get("content", []) or []:
        if c.get("type") == "text" and "text" in c:
            parts.append(c["text"])
    return "\n".join(parts).strip()


# ---- NEU: Parser für EINEN Request pro Notruf (custom_id = Notruf_12) ----
def parse_anthropic_batch_results_jsonl_to_df(jsonl_text: str) -> pd.DataFrame:
    by_call = {}

    for line in jsonl_text.splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)

        call_id = rec.get("custom_id")  # jetzt: "Notruf_12"
        if not call_id:
            continue

        by_call.setdefault(call_id, {"file": call_id})

        result = rec.get("result") or {}
        rtype = result.get("type")

        if rtype != "succeeded":
            for d in DOMAINS:
                by_call[call_id][f"severity_{d}"] = None
                #by_call[call_id][f"confidence_{d}"] = None
                #by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
                #by_call[call_id][f"error_{d}"] = json.dumps(result, ensure_ascii=False)
            continue

        msg = result.get("message") or {}
        text = _extract_text_from_anthropic_message(msg)

        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            for d in DOMAINS:
                by_call[call_id][f"severity_{d}"] = None
              #  by_call[call_id][f"confidence_{d}"] = None
             #   by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
             #   by_call[call_id][f"error_{d}"] = f"parse_error: no_json_in_text: {text[:200]}"
            continue

        try:
            data = json.loads(m.group(0))  # erwartet: {"A": {...}, ...}
        except Exception as e:
            for d in DOMAINS:
                by_call[call_id][f"severity_{d}"] = None
               # by_call[call_id][f"confidence_{d}"] = None
              #  by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
              #  by_call[call_id][f"error_{d}"] = f"parse_error: {e}"
            continue

        for d in DOMAINS:
            block = data.get(d)
            if not isinstance(block, dict):
                by_call[call_id][f"severity_{d}"] = None
                #by_call[call_id][f"confidence_{d}"] = None
               # by_call[call_id][f"findings_{d}"] = json.dumps([], ensure_ascii=False)
                #by_call[call_id][f"error_{d}"] = f"missing_or_invalid_block_{d}"
                continue

            sev = block.get("severity")
   

            by_call[call_id][f"severity_{d}"] = sev
            

    df = pd.DataFrame(list(by_call.values()))
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    cols = ["file", "call_nr"]
    for d in DOMAINS:
        cols += [f"severity_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


def run_abcde_batch_experiment_anthropic(
    model: str,
    temperature: float,
    *,
    calls_dir: Path,
    guidelines_dir: Path,      # enthält jetzt GENAU 1 Guideline-Datei
    results_dir: Path,
    max_tokens: int = 800,     # größer, weil A..E in einem Output
    poll_every_seconds: int = 30,
    output_contract: str = OUTPUT_CONTRACT_DEFAULT,
):
    """
    Claude Message Batches (Anthropic):
    - PRO NOTRUF genau 1 Request
    - guidelines_dir enthält GENAU 1 Master-Guideline
    - Output ist JSON mit A..E-Blöcken
    - Ergebnis wird als JSONL gespeichert und als CSV (wide) exportiert
    """
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise EnvironmentError("ANTHROPIC_API_KEY fehlt in env.")
    client = Anthropic(api_key=api_key)

    calls_dir = Path(calls_dir).resolve()
    guidelines_dir = Path(guidelines_dir).resolve()
    results_dir = Path(results_dir).resolve()
    results_dir.mkdir(parents=True, exist_ok=True)

    call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
    if not call_files:
        raise FileNotFoundError(f"Keine Notrufe gefunden in {calls_dir}")

    prompt = ("You are an expert in analysing emergency calls. \n"
              "You should evaluate the call transcript according the ABCDE schema. \n"
              "ABCDE Schema: Airway, Breathing, Circulation, Disability, Exposure.\n"
              "Give every category a severity score from 0 to 3.\n"
              "**0** – No information available; no conclusions possible\n"  
            "**1** – No or only minor impairment suspected\n"  
            "**2** – Significant impairment present; no acute life threat suspected (ambulance indicated)\n"  
            "**3** – Severe impairment; vital threat possible or likely; immediate intervention required (urgent ambulance, possibly physician response unit)")

    system_prompt = f"{prompt}\n\n{output_contract}"

    # ---- NEU: Batch requests bauen: 1 Request pro Notruf ----
    batch_requests = []
    for fp in call_files:
        transcript = fp.read_text(encoding="utf-8").strip()
        batch_requests.append({
            "custom_id": f"{fp.stem}",  # <-- nur Notruf_12
            "params": {
                "model": model,
                "temperature": float(temperature),
                "max_tokens": int(max_tokens),
                "system": system_prompt,
                "messages": [
                    {"role": "user", "content": transcript}
                ],
            }
        })

    message_batch = client.messages.batches.create(requests=batch_requests)

    # warten bis ended
    while True:
        mb = client.messages.batches.retrieve(message_batch.id)
        if mb.processing_status == "ended":
            message_batch = mb
            break
        print(f"[{message_batch.id}] processing_status={mb.processing_status} ...")
        time.sleep(poll_every_seconds)

    if not message_batch.results_url:
        raise RuntimeError(f"Batch ended, aber results_url fehlt: {message_batch}")

    # results_url downloaden (JSONL)
    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
    }
    r = requests.get(message_batch.results_url, headers=headers, timeout=300)
    r.raise_for_status()
    results_text = r.text

    # speichern
    out_jsonl = results_dir / f"batch_output_{message_batch.id}.jsonl"
    out_jsonl.write_text(results_text, encoding="utf-8")

    df = parse_anthropic_batch_results_jsonl_to_df(results_text)
    out_csv = next_free_run_path(results_dir, model=model, temperature=temperature)
    df.to_csv(out_csv, index=False, encoding="utf-8")

    return {
        "batch_id": message_batch.id,
        "processing_status": message_batch.processing_status,
        "results_url": message_batch.results_url,
        "output_jsonl_path": str(out_jsonl),
        "csv_path": str(out_csv),
        "df": df,
    }


In [ ]:
from pathlib import Path

res = run_abcde_batch_experiment_anthropic(
    model="claude-sonnet-4-6",   # Sonnet 4.x Modellname :contentReference[oaicite:10]{index=10}
    temperature=0,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    max_tokens=400,
)

print(res["csv_path"])
res["df"].head()

In [ ]:
from pathlib import Path

res = run_abcde_batch_experiment_anthropic(
    model="claude-sonnet-4-6",   # Sonnet 4.x Modellname :contentReference[oaicite:10]{index=10}
    temperature=0.5,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    max_tokens=400,
)

print(res["csv_path"])
res["df"].head()

In [ ]:
from pathlib import Path

res = run_abcde_batch_experiment_anthropic(
    model="claude-sonnet-4-6",   # Sonnet 4.x Modellname :contentReference[oaicite:10]{index=10}
    temperature=1,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    max_tokens=400,
)

print(res["csv_path"])
res["df"].head()

In [ ]:
print(res)